# modelo_opensees_3d — notebook de análisis

Versión ordenada y ejecutable del módulo `scripts/modelo_opensees_3d.py`. Construye el modelo 3D de vigas, columnas, muros ShellMITC4 y diafragmas rígidos.

## Preparación y generación de entrada

In [ ]:
from pathlib import Path
import subprocess, sys, json
import openseespy.opensees as ops
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
SCRIPTS=ROOT/'scripts'; OUT=ROOT/'outputs'
r=subprocess.run([sys.executable,str(SCRIPTS/'generar_modelo_manual.py')],cwd=ROOT,text=True,capture_output=True)
print(r.stdout)
if r.returncode: print(r.stderr); raise RuntimeError('No se pudo generar el modelo')

## 1. Construcción del modelo OpenSees 3D

In [ ]:
sys.path.insert(0,str(SCRIPTS))
import modelo_opensees_3d as m3d
data=m3d.build_model()
print('Nodos estructurales:',len(data['structural_node_ids']))
print('Elementos de entrada:',len(data['elements']))
print('Losas:',len(data.get('slabs',[])))
print('Muros ShellMITC4:',data.get('wall_mesh',{}).get('shell_count',0))
print('Nodos OpenSees:',len(ops.getNodeTags()))
print('Elementos OpenSees:',len(ops.getEleTags()))

## 2. Cargas y análisis gravitacional

In [ ]:
m3d.apply_slab_loads_to_walls(data)
m3d.apply_slab_loads_to_beams(data)
m3d.apply_wall_self_weight(data.get('wall_mesh',{}))
ok_gravity=m3d.analyze_gravity(data)
print('Equilibrio gravitacional:', 'OK' if ok_gravity else 'REVISAR')

## 3. Nodos coincidentes y diafragmas rígidos

In [ ]:
ties=m3d.merge_coincident_nodes(data)
diaphragms=m3d.apply_rigid_diaphragms(data)
print('Vínculos de nodos coincidentes:',len(ties))
print('Diafragmas creados:',len(diaphragms))
for d in diaphragms: print(d['subbuilding'], 'z=',d['z_m'],'master=',d['master_node'],'slaves=',d['slave_count'])

## 4. Verificación de compatibilidad lateral

In [ ]:
ok_diaphragm=m3d.verify_diaphragm_compatibility(data)
print('Compatibilidad:', 'OK' if ok_diaphragm else 'REVISAR')

## Ejecución directa

Desde `P1L2`:

```powershell
python scripts\generar_modelo_manual.py
python scripts\modelo_opensees_3d.py
```